# [B3] Which STTR translational outcomes survive program stratification?

**Status:** exploratory — non-citable  
**Research questions:** B3 — **Phase II → III survival probability**, **Phase II → III latency**, and **Transition effectiveness rate** ([`docs/research-questions.md`](../../docs/research-questions.md#b3-inferential-tier-3))  
**Decision this informs:** whether the current repository can report an STTR-specific outcome now, and which program fields must be restored before any STTR Phase II→III analysis.  
**Data as of:** SBIR.gov bulk export retrieved 2026-10-21; pinned below by SHA-256 and byte count.  
**Owner:** STTR translational-outcomes workbench

The answer is split deliberately. The source award history preserves `Program`, so it can support a bounded description of same-firm STTR Phase I→later Phase II awards. The current Phase II, Phase III, pair, score, and survival marts do not preserve enough program information for an STTR-specific Phase II→III result. This notebook does not silently substitute a pooled SBIR/STTR number.

A later award to the same resolved firm is an **observed funding sequence**. It is not evidence that the Phase II project derives from the Phase I project, not statutory Phase III, and not a causal effect of STTR.


## Data contract

- **Population:** physical rows in the pinned SBIR.gov export whose `Program` is exactly `STTR` or `SBIR`. The progression denominator is one first observed STTR Phase I per resolved firm.
- **Primary outcome:** an observed same-firm `Program = STTR`, `Phase = Phase II` award with award-year gap ≤ 2, 3, or 5 among firms mature for the corresponding horizon. Later SBIR Phase II awards are reported separately and never count as STTR progression.
- **Grain:** `EXPORT_ROW_V1` for source diagnostics; stable SBIR award grain for deduplication; exact firm component for progression.
- **Keys:** `sbir-source-v2` award key; firm identity is the existing exact connected component across canonically normalized, valid UEI/DUNS and `ORGANIZATION_KEY_V1` name aliases used by the agency-private-capital outcome code. Invalid and sentinel identifiers are missing before resolution. Components with multiple distinct valid UEIs or DUNSes are audited and excluded from the primary progression description, with an inclusion sensitivity reported alongside it. This is exact alias resolution, not fuzzy matching.
- **Time:** `Award Year` is the complete source time field. Same-year order counts only when both proposal-award dates are present, coherent with `Award Year`, and show Phase II strictly after Phase I. Same-day records are reported separately and are not later awards.
- **Right censoring:** a horizon enters the denominator only after the full calendar year containing that horizon has ended by 2026-10-21. Recent firms remain under observation; they are not failures.
- **Inputs:** one pinned SBIR.gov export supplied with `SBIR_STTR_AWARDS_PATH`; optional local transition Parquets are inspected only for schema readiness. No network call or generator runs here.
- **Exclusions:** unresolved or identifier-conflicted firm components, missing award years, same-day or out-of-order Phase II records, and ambiguous same-year order. Phase labels follow the canonical exact-phase rule: suffix-coded values normalize to missing, so the pinned snapshot's 3,303 SBIR rows labelled `Phase II – Direct to Phase II` cannot enter either Phase II path.
- **Missingness:** an absent input yields an empty, labelled result. No missing artifact or missing match is converted to zero evidence.
- **Assumptions:** exact alias co-occurrence denotes the same firm; it does not establish project lineage, research-institution involvement, technical similarity, or commercialization.
- **Seed:** `20261028`; no stochastic operation is used.
- **Outputs:** in-notebook diagnostic tables only. Exploratory and non-citable.


In [ ]:
from __future__ import annotations

import os
from datetime import date
from pathlib import Path

import pandas as pd

from sbir_analytics.assets.agency_private_capital.outcomes import _resolve_company_identities
from sbir_etl.extractors.sbir_award_export import (
    SBIR_GOV_SOURCE_COLUMNS,
    normalize_sbir_phase,
    read_sbir_gov_csv,
)
from sbir_etl.identity import sbir_award_grain_key
from sbir_etl.models.phase_transition import (
    PhaseIIAward,
    PhaseIIIContract,
    PhaseTransitionPair,
    PhaseTransitionSurvival,
)
from sbir_etl.utils.data.file_io import file_sha256
from sbir_etl.utils.identifiers import normalize_duns, normalize_uei


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


def portable_path(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(REPO_ROOT.resolve()))
    except ValueError:
        return f"<external>/{path.name}"


REPO_ROOT = find_repo_root()
REPO_ROOT


In [ ]:
AS_OF_DATE = date(2026, 10, 21)
SOURCE_URL = "https://data.www.sbir.gov/mod_awarddatapublic/award_data.csv"
SOURCE_RETRIEVED_DATE = "2026-10-21"
EXPECTED_AWARDS_SHA256 = "1058456da366796402ad5264bbd90cfcd0e1de6dc8206db372579e06de27c600"
EXPECTED_AWARDS_BYTES = 395_208_648
RANDOM_SEED = 20261028  # declared for reproducibility; no random operation is used
FOLLOW_UP_HORIZONS = (2, 3, 5)
PRIMARY_HORIZON = 5

override = os.environ.get("SBIR_STTR_AWARDS_PATH")
if override:
    AWARDS_PATH = Path(override).expanduser()
else:
    award_candidates = (
        REPO_ROOT
        / "data"
        / "raw"
        / "sbir"
        / "award-export"
        / SOURCE_RETRIEVED_DATE
        / "award_data.csv",
        REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv",
    )
    AWARDS_PATH = next((path for path in award_candidates if path.is_file()), award_candidates[0])

MART_PATHS = {
    "enriched award spine": REPO_ROOT / "data" / "processed" / "enriched_sbir_awards.parquet",
    "transition scores": REPO_ROOT / "data" / "processed" / "transitions.parquet",
    "Phase II mart": REPO_ROOT / "data" / "processed" / "phase_ii_awards.parquet",
    "Phase III mart": REPO_ROOT / "data" / "processed" / "phase_iii_contracts.parquet",
    "Phase II–III pairs": REPO_ROOT / "data" / "processed" / "phase_ii_iii_pairs.parquet",
    "survival mart": REPO_ROOT / "data" / "processed" / "phase_transition_survival.parquet",
}


In [ ]:
awards_exists = AWARDS_PATH.is_file()
observed_awards_bytes = AWARDS_PATH.stat().st_size if awards_exists else None
observed_awards_sha256 = file_sha256(AWARDS_PATH) if awards_exists else None
source_verified = bool(
    awards_exists
    and observed_awards_bytes == EXPECTED_AWARDS_BYTES
    and observed_awards_sha256 == EXPECTED_AWARDS_SHA256
)

if source_verified:
    canonical_awards_source = read_sbir_gov_csv(AWARDS_PATH)
    literal_na_cells = int(canonical_awards_source.eq("N/A").sum().sum())
    assert literal_na_cells > 0
else:
    canonical_awards_source = pd.DataFrame(columns=SBIR_GOV_SOURCE_COLUMNS)
    literal_na_cells = pd.NA

input_rows = [
    {
        "input": "SBIR.gov award export",
        "path": portable_path(AWARDS_PATH),
        "exists": awards_exists,
        "pinned_bytes_match": (
            observed_awards_bytes == EXPECTED_AWARDS_BYTES if awards_exists else pd.NA
        ),
        "pinned_sha256_match": (
            observed_awards_sha256 == EXPECTED_AWARDS_SHA256
            if awards_exists
            else pd.NA
        ),
        "literal N/A cells preserved": literal_na_cells,
    }
]
input_rows.extend(
    {
        "input": name,
        "path": portable_path(path),
        "exists": path.is_file(),
        "pinned_bytes_match": pd.NA,
        "pinned_sha256_match": pd.NA,
        "literal N/A cells preserved": pd.NA,
    }
    for name, path in MART_PATHS.items()
)
input_status = pd.DataFrame(input_rows)
if not awards_exists:
    print("Pinned awards input absent — set SBIR_STTR_AWARDS_PATH; analytical tables stay empty.")
elif not source_verified:
    print(
        "Awards input does not match the pinned 2026-10-21 bytes; "
        "analytical tables are withheld."
    )
input_status


## Program-preservation audit

The question is not whether a program label existed upstream. It is whether the artifact carrying the outcome still contains that label. A downstream join is not assumed safe merely because an upstream table once had `Program`.

The Phase III source code distinguishes FPDS `SR3` (SBIR) from `ST3` (STTR), but the `PhaseIIIContract` model retains neither code nor program. Other Phase III identification methods do not necessarily carry either code, so future program provenance must be nullable rather than inferred. The Phase II model, pair model, score output, and survival model also omit program. Therefore these marts cannot be filtered to STTR on their own.


In [ ]:
def parquet_columns(path: Path) -> set[str] | None:
    if not path.is_file():
        return None
    try:
        import pyarrow.parquet as pq

        return set(pq.read_schema(path).names)
    except Exception as exc:
        print(f"Could not inspect {portable_path(path)}: {exc}")
        return None


declared_schemas = {
    "SBIR.gov export": set(SBIR_GOV_SOURCE_COLUMNS),
    "enriched award spine": {"award_id", "program", "phase"},
    "transition scores": {
        "award_id",
        "contract_id",
        "score",
        "method",
        "signals",
        "computed_at",
    },
    "Phase II mart": set(PhaseIIAward.model_fields),
    "Phase III mart": set(PhaseIIIContract.model_fields),
    "Phase II–III pairs": set(PhaseTransitionPair.model_fields),
    "survival mart": set(PhaseTransitionSurvival.model_fields),
}
safety = {
    "SBIR.gov export": "yes — filter Program before analysis",
    "enriched award spine": "yes — only before downstream program loss",
    "transition scores": "no — output drops program; audited award-key join required",
    "Phase II mart": "no — PhaseIIAward drops program",
    "Phase III mart": "no — program absent; SR/ST evidence exists for only some upstream methods",
    "Phase II–III pairs": "no — neither side's program is retained",
    "survival mart": "no — Phase II program is absent",
}
audit_rows = []
for artifact, columns in declared_schemas.items():
    declared_has_program = bool({column.lower() for column in columns} & {"program"})
    runtime_columns = None
    if artifact in MART_PATHS:
        runtime_columns = parquet_columns(MART_PATHS[artifact])
    elif artifact == "SBIR.gov export" and source_verified:
        runtime_columns = set(canonical_awards_source.columns)
    runtime_has_program = (
        bool({column.lower() for column in runtime_columns} & {"program"})
        if runtime_columns is not None
        else pd.NA
    )
    audit_rows.append(
        {
            "artifact": artifact,
            "declares program": declared_has_program,
            "runtime artifact present": (artifact == "SBIR.gov export" and source_verified)
            or (artifact in MART_PATHS and MART_PATHS[artifact].is_file()),
            "runtime program field": runtime_has_program,
            "STTR-safe use now": safety[artifact],
        }
    )
program_preservation_audit = pd.DataFrame(audit_rows)
program_preservation_audit


## Source-level STTR award spine

The canonical lossless reader first validates the ordered 42-column header and every physical row width while preserving exact source strings, including literal `N/A`. Only then does this notebook project the fields needed for identity, award grain, program, phase, agency, and time. A missing or byte-mismatched source returns an empty frame instead of silently selecting another vintage.


In [ ]:
SOURCE_PROJECTION = [
    "Company",
    "Agency",
    "Branch",
    "Phase",
    "Program",
    "Agency Tracking Number",
    "Contract",
    "Proposal Award Date",
    "Contract End Date",
    "Solicitation Number",
    "Award Year",
    "UEI",
    "Duns",
]

if not source_verified:
    awards_raw = pd.DataFrame(columns=["_source_row", *SOURCE_PROJECTION])
else:
    awards_raw = canonical_awards_source.loc[:, SOURCE_PROJECTION].copy()
    awards_raw.insert(0, "_source_row", range(2, len(awards_raw) + 2))
    print(f"Loaded {len(awards_raw):,} physical export rows from the pinned source.")
del canonical_awards_source
awards_raw.head()


In [ ]:
DUNS_SENTINELS = {"000000000", "111111111", "999999999"}
UEI_SENTINELS = {"000000000000", "111111111111", "999999999999", "XXXXXXXXXXXX"}


def normalize_agency_label(value: object) -> str | None:
    if value is None or pd.isna(value):
        return None
    normalized = " ".join(str(value).split()).upper()
    return normalized or None


def normalize_identity_identifiers(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply the notebook's canonical identifier and sentinel policy."""

    normalized = frame.copy()
    normalized["uei_canonical"] = normalized["UEI"].map(normalize_uei)
    normalized["duns_canonical"] = normalized["Duns"].map(normalize_duns)
    normalized["uei_is_sentinel"] = normalized["uei_canonical"].isin(UEI_SENTINELS)
    normalized["duns_is_sentinel"] = normalized["duns_canonical"].isin(DUNS_SENTINELS)
    normalized["uei"] = normalized["uei_canonical"].mask(normalized["uei_is_sentinel"])
    normalized["duns"] = normalized["duns_canonical"].mask(normalized["duns_is_sentinel"])
    return normalized


awards = awards_raw.copy()
awards["program"] = awards["Program"].fillna("").astype(str).str.strip().str.upper()
awards["phase_label"] = awards["Phase"].map(normalize_sbir_phase)
awards["award_year"] = pd.to_numeric(awards["Award Year"], errors="coerce").astype("Int64")
awards["award_date"] = pd.to_datetime(
    awards["Proposal Award Date"], errors="coerce", format="mixed"
)
awards["date_year_coherent"] = awards["award_date"].notna() & (
    awards["award_date"].dt.year.astype("Int64") == awards["award_year"]
)
awards["company_name"] = awards["Company"]
awards["agency_normalized"] = awards["Agency"].map(normalize_agency_label)
awards = normalize_identity_identifiers(awards)

if awards.empty:
    program_phase_counts = pd.DataFrame()
else:
    program_phase_counts = pd.crosstab(
        awards["program"].replace("", "(missing)"),
        awards["Phase"].fillna("(missing)"),
        margins=True,
    )
program_phase_counts


In [ ]:
if awards.empty:
    source_coverage = pd.DataFrame()
else:
    sttr_rows = awards.loc[awards["program"].eq("STTR")]
    source_coverage = pd.DataFrame(
        [
            {"measure": "physical export rows", "value": len(awards)},
            {"measure": "STTR rows", "value": len(sttr_rows)},
            {"measure": "STTR Phase I rows", "value": int(sttr_rows["phase_label"].eq("I").sum())},
            {
                "measure": "STTR Phase II rows",
                "value": int(sttr_rows["phase_label"].eq("II").sum()),
            },
            {
                "measure": "STTR Phase III rows",
                "value": int(sttr_rows["phase_label"].eq("III").sum()),
            },
            {
                "measure": "STTR rows with Award Year",
                "value": int(sttr_rows["award_year"].notna().sum()),
            },
            {
                "measure": "STTR rows with coherent exact date",
                "value": int(sttr_rows["date_year_coherent"].sum()),
            },
        ]
    )
source_coverage


## Exact identity, award grain, and cross-program firms

Resolve identity across the complete SBIR/STTR snapshot before filtering outcomes. UEI and DUNS first pass through the canonical repository normalizers; malformed values (including raw DUNS `0`) and explicit canonical placeholders (all-zero, all-one, or all-nine identifiers, plus X-only UEI) become missing before the alias graph is built. This lets a valid UEI-only row connect to an older valid DUNS-only row through an observed alias without letting a shared sentinel identifier join unrelated firms. It also makes cross-program firms visible. The within-STTR numerator still requires a later STTR Phase II; a later SBIR Phase II is a separate category.

The imported resolver is an internal helper from the current agency-private-capital implementation. Promotion would require a public, versioned firm-component primitive. This notebook does not create a competing identity implementation. Exact duplicate `(name, UEI, DUNS)` triples are collapsed before resolution and mapped back afterward: duplicate graph edges cannot change connected components, and this avoids repeating the resolver's row traversal for identical evidence. Components containing multiple distinct valid UEIs or DUNSes are surfaced below and excluded from the primary progression description rather than treated as unambiguous firms.

After resolving the full identity snapshot, stable award keys are computed only for STTR Phase I rows and Phase II rows belonging to a firm component with an STTR Phase I. Rows outside that set cannot enter either the primary or sensitivity progression result; pruning them avoids a second per-row key pass over irrelevant SBIR awards while retaining the canonical `sbir-source-v2` key primitive.


In [ ]:
def raw_identifier_present(values: pd.Series) -> pd.Series:
    return values.notna() & values.astype("string").str.strip().ne("").fillna(False)


def raw_zero_like(value: object) -> bool:
    if value is None or pd.isna(value):
        return False
    cleaned = "".join(character for character in str(value) if character.isalnum()).upper()
    return bool(cleaned) and set(cleaned) == {"0"}


identifier_quality = pd.DataFrame(
    [
        {
            "identifier": "UEI",
            "raw nonblank": int(raw_identifier_present(awards["UEI"]).sum()),
            "canonical invalid": int(
                (raw_identifier_present(awards["UEI"]) & awards["uei_canonical"].isna()).sum()
            ),
            "sentinel removed": int(awards["uei_is_sentinel"].sum()),
            "raw zero-like (included in invalid/sentinel)": int(
                awards["UEI"].map(raw_zero_like).sum()
            ),
            "usable canonical IDs": int(awards["uei"].notna().sum()),
        },
        {
            "identifier": "DUNS",
            "raw nonblank": int(raw_identifier_present(awards["Duns"]).sum()),
            "canonical invalid": int(
                (raw_identifier_present(awards["Duns"]) & awards["duns_canonical"].isna()).sum()
            ),
            "sentinel removed": int(awards["duns_is_sentinel"].sum()),
            "raw zero-like (included in invalid/sentinel)": int(
                awards["Duns"].map(raw_zero_like).sum()
            ),
            "usable canonical IDs": int(awards["duns"].notna().sum()),
        },
    ]
).set_index("identifier")

# Regression guard: raw sentinel variants traverse the same normalization helper
# as the export. Without its sentinel mask, both rows would share duns:000000000.
sentinel_probe = normalize_identity_identifiers(
    pd.DataFrame(
        {
            "company_name": ["SYNTHETIC SENTINEL ALPHA", "SYNTHETIC SENTINEL BETA"],
            "UEI": [None, None],
            "Duns": ["000000000", "000-000-000"],
            "phase_label": ["I", "II"],
            "award_year": [2020, 2021],
        }
    )
)
assert sentinel_probe["duns"].isna().all()
assert normalize_duns("0") is None
sentinel_probe_keys, sentinel_probe_aliases = _resolve_company_identities(
    sentinel_probe[["company_name", "uei", "duns"]]
)
sentinel_progression_candidates = int(sentinel_probe_keys[0] == sentinel_probe_keys[1])
assert sentinel_progression_candidates == 0
assert not any(alias.startswith("duns:") for aliases in sentinel_probe_aliases for alias in aliases)

awards["firm_key"] = pd.Series(pd.NA, index=awards.index, dtype="object")
identity_rows = awards.loc[awards["program"].isin({"SBIR", "STTR"})].drop(
    columns="firm_key"
).copy()
identity_columns = ["company_name", "uei", "duns"]
identity_patterns = identity_rows[identity_columns].drop_duplicates().copy()
if not identity_patterns.empty:
    firm_keys, _ = _resolve_company_identities(
        identity_patterns[identity_columns]
    )
    identity_patterns["firm_key"] = firm_keys
    identity_rows = (
        identity_rows.reset_index(names="_award_index")
        .merge(
            identity_patterns,
            on=identity_columns,
            how="left",
            sort=False,
            validate="many_to_one",
        )
    )
    awards.loc[identity_rows["_award_index"], "firm_key"] = identity_rows["firm_key"].to_numpy()
if identity_rows.empty:
    identity_component_audit = pd.DataFrame()
    identity_component_conflicts = pd.DataFrame()
    conflicted_firm_keys: set[str] = set()
else:
    identity_component_audit = (
        identity_rows.loc[identity_rows["firm_key"].notna()]
        .groupby("firm_key")
        .agg(
            rows=("firm_key", "size"),
            programs=("program", lambda values: tuple(sorted(set(values)))),
            valid_ueis=("uei", lambda values: tuple(sorted(set(values.dropna())))),
            valid_duns=("duns", lambda values: tuple(sorted(set(values.dropna())))),
            source_company_labels=("Company", lambda values: values.dropna().nunique()),
        )
    )
    identity_component_audit["valid UEI count"] = identity_component_audit["valid_ueis"].map(len)
    identity_component_audit["valid DUNS count"] = identity_component_audit["valid_duns"].map(len)
    identity_component_conflicts = identity_component_audit.loc[
        identity_component_audit["valid UEI count"].gt(1)
        | identity_component_audit["valid DUNS count"].gt(1)
    ].sort_values(["rows", "valid UEI count", "valid DUNS count"], ascending=False)
    conflicted_firm_keys = set(identity_component_conflicts.index)

sttr_identity = awards.loc[awards["program"].eq("STTR")].copy()
identity_coverage = pd.DataFrame(
    [
        {
            "population": "STTR export rows",
            "rows": len(sttr_identity),
            "resolved rows": int(sttr_identity["firm_key"].notna().sum()),
            "firm components": int(sttr_identity["firm_key"].nunique()),
            "usable UEI present": int(sttr_identity["uei"].notna().sum()),
            "usable DUNS present": int(sttr_identity["duns"].notna().sum()),
            "conflicted components excluded from progression": int(
                sttr_identity.loc[
                    sttr_identity["firm_key"].isin(conflicted_firm_keys), "firm_key"
                ].nunique()
            ),
        }
    ]
)
identifier_quality


In [ ]:
identity_conflict_summary = pd.DataFrame(
    [
        {
            "SBIR/STTR rows": len(identity_rows),
            "distinct identity triples traversed by resolver": len(identity_patterns),
            "duplicate identity rows skipped by resolver": (
                len(identity_rows) - len(identity_patterns)
            ),
            "resolved components": len(identity_component_audit),
            "components with multiple valid UEIs and/or DUNSes": len(identity_component_conflicts),
            "source rows in those components": int(identity_component_conflicts["rows"].sum())
            if not identity_component_conflicts.empty
            else 0,
            "synthetic sentinel Phase I→II candidates": sentinel_progression_candidates,
            "primary progression policy": "exclude conflicted components",
        }
    ]
)
display(identity_coverage, identity_conflict_summary, identity_component_conflicts.head(20))


In [ ]:
if awards.empty:
    analytical_awards = awards.copy()
    grain_audit = pd.DataFrame()
else:
    sttr_phase_i_mask = (
        awards["program"].eq("STTR")
        & awards["phase_label"].eq("I")
        & awards["firm_key"].notna()
    )
    sttr_phase_i_firm_keys = set(awards.loc[sttr_phase_i_mask, "firm_key"])
    phase_ii_mask = awards["program"].isin({"SBIR", "STTR"}) & awards["phase_label"].eq("II")
    phase_ii_in_scope = phase_ii_mask & awards["firm_key"].isin(sttr_phase_i_firm_keys)
    relevant = awards.loc[sttr_phase_i_mask | phase_ii_in_scope].copy()
    relevant["award_key"] = [
        sbir_award_grain_key(row) for row in relevant.to_dict(orient="records")
    ]
    analytical_awards = (
        relevant.sort_values(["award_key", "_source_row"], kind="stable")
        .drop_duplicates("award_key", keep="first")
        .reset_index(drop=True)
    )
    grain_audit = pd.DataFrame(
        [
            {
                "potential key rows before firm pruning": int(
                    sttr_phase_i_mask.sum() + phase_ii_mask.sum()
                ),
                "in-scope export rows keyed": len(relevant),
                "Phase II rows pruned (no resolved STTR Phase I firm)": int(
                    (phase_ii_mask & ~phase_ii_in_scope).sum()
                ),
                "stable award keys": relevant["award_key"].nunique(),
                "source-edition/duplicate rows collapsed": (
                    len(relevant) - relevant["award_key"].nunique()
                ),
            }
        ]
    )
grain_audit


In [ ]:
if awards.empty:
    firm_program_summary = pd.DataFrame()
    cross_program_firms: set[str] = set()
else:
    firm_programs = (
        awards.loc[awards["firm_key"].notna() & awards["program"].isin({"SBIR", "STTR"})]
        .groupby("firm_key")["program"]
        .agg(lambda values: tuple(sorted(set(values))))
    )
    cross_program_firms = set(
        firm_programs.loc[firm_programs.map(lambda values: values == ("SBIR", "STTR"))].index
    )
    firm_program_summary = (
        firm_programs.map(lambda values: " + ".join(values))
        .value_counts()
        .rename_axis("programs observed anywhere in snapshot")
        .rename("firm components")
        .to_frame()
    )
firm_program_summary


## Descriptive Phase I→II progression

Anchor each non-conflicted firm component on its first observed STTR Phase I. Search later same-firm Phase II awards. Require STTR on the outcome row for the within-program measure, and retain SBIR outcomes as a separate path. This avoids counting a cross-program firm's SBIR Phase II as STTR progression.

A same-year pair is eligible only when coherent exact dates show Phase II strictly after Phase I. Same-day, reverse-order, and date-ambiguous relations are reported below but do not count as later. A later award year establishes order but not project lineage.

Every horizon result is labelled **award-year gap ≤ N among N-horizon-mature firms**. The mature cohort changes with N, so cross-horizon differences combine a longer follow-up window with different vintage composition; they are not a cumulative curve for one fixed cohort. This is a complete-source descriptive view, so no sampling interval is attached.

Multi-ID components are disproportionately long-lived and award-rich, so excluding them may select on the opportunity to observe a Phase II and can move the share in either direction. Each horizon therefore reports a primary row excluding those ambiguous components and an inclusion sensitivity row; the sensitivity exposes this measurement choice but does not establish which identity interpretation is correct.


In [ ]:
def build_progression_frames(
    excluded_keys: set[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if analytical_awards.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    phase_i = analytical_awards.loc[
        analytical_awards["program"].eq("STTR")
        & analytical_awards["phase_label"].eq("I")
        & analytical_awards["firm_key"].notna()
        & ~analytical_awards["firm_key"].isin(excluded_keys)
        & analytical_awards["award_year"].notna()
    ].copy()
    phase_i["_date_sort"] = phase_i["award_date"].where(
        phase_i["date_year_coherent"], pd.Timestamp.max
    )
    frame_anchors = (
        phase_i.sort_values(
            ["firm_key", "award_year", "_date_sort", "award_key"], kind="stable"
        )
        .drop_duplicates("firm_key", keep="first")
        [
            [
                "firm_key",
                "award_key",
                "award_year",
                "award_date",
                "date_year_coherent",
                "Agency",
                "agency_normalized",
            ]
        ]
        .rename(
            columns={
                "award_key": "phase_i_award_key",
                "award_year": "phase_i_year",
                "award_date": "phase_i_date",
                "date_year_coherent": "phase_i_date_coherent",
                "Agency": "phase_i_agency",
                "agency_normalized": "phase_i_agency_normalized",
            }
        )
        .reset_index(drop=True)
    )
    frame_anchors["cross_program_firm"] = frame_anchors["firm_key"].isin(cross_program_firms)

    phase_ii = analytical_awards.loc[
        analytical_awards["phase_label"].eq("II")
        & analytical_awards["program"].isin({"SBIR", "STTR"})
        & analytical_awards["firm_key"].notna()
        & ~analytical_awards["firm_key"].isin(excluded_keys)
        & analytical_awards["award_year"].notna()
    ][
        [
            "firm_key",
            "award_key",
            "award_year",
            "award_date",
            "date_year_coherent",
            "Agency",
            "agency_normalized",
            "program",
        ]
    ].rename(
        columns={
            "award_key": "phase_ii_award_key",
            "award_year": "phase_ii_year",
            "award_date": "phase_ii_date",
            "date_year_coherent": "phase_ii_date_coherent",
            "Agency": "phase_ii_agency",
            "agency_normalized": "phase_ii_agency_normalized",
            "program": "phase_ii_program",
        }
    )
    joined = frame_anchors.merge(phase_ii, on="firm_key", how="left")
    distinct_awards = joined["phase_ii_award_key"].notna() & joined["phase_ii_award_key"].ne(
        joined["phase_i_award_key"]
    )
    same_year = joined["phase_ii_year"].eq(joined["phase_i_year"]) & distinct_awards
    dates_comparable = (
        same_year & joined["phase_i_date_coherent"] & joined["phase_ii_date_coherent"]
    )
    strict_same_year_later = dates_comparable & joined["phase_ii_date"].gt(
        joined["phase_i_date"]
    )
    same_day = dates_comparable & joined["phase_ii_date"].eq(joined["phase_i_date"])
    reverse_order = dates_comparable & joined["phase_ii_date"].lt(joined["phase_i_date"])
    ambiguous_same_year = same_year & ~dates_comparable
    later_year = joined["phase_ii_year"].gt(joined["phase_i_year"]) & distinct_awards
    frame_relations = joined.loc[later_year | strict_same_year_later].copy()
    frame_relations["award_year_gap"] = (
        frame_relations["phase_ii_year"] - frame_relations["phase_i_year"]
    ).astype("Int64")

    frame_order_audit = pd.DataFrame(
        [
            {
                "same-year relation": label,
                "relations": int(mask.sum()),
                "Phase I anchors affected": int(joined.loc[mask, "phase_i_award_key"].nunique()),
                "counts as later": counts_as_later,
            }
            for label, mask, counts_as_later in (
                ("strictly later exact date", strict_same_year_later, True),
                ("same exact date", same_day, False),
                ("reverse exact-date order", reverse_order, False),
                ("date missing/incoherent", ambiguous_same_year, False),
            )
        ]
    )
    return frame_anchors, frame_relations, frame_order_audit


potential_phase_i = analytical_awards.loc[
    analytical_awards["program"].eq("STTR")
    & analytical_awards["phase_label"].eq("I")
    & analytical_awards["firm_key"].notna()
    & analytical_awards["award_year"].notna()
]
excluded_conflicted_anchor_components = potential_phase_i.loc[
    potential_phase_i["firm_key"].isin(conflicted_firm_keys), "firm_key"
].nunique()
anchors, candidate_relations, same_year_order_audit = build_progression_frames(
    conflicted_firm_keys
)
sensitivity_anchors, sensitivity_candidate_relations, _ = build_progression_frames(set())

print(f"Primary first-observed STTR Phase I firm anchors: {len(anchors):,}")
print(f"Sensitivity anchors including multi-ID components: {len(sensitivity_anchors):,}")
print(
    "Conflicted firm components excluded from primary: "
    f"{excluded_conflicted_anchor_components:,}"
)
same_year_order_audit


In [ ]:
def mature_for_horizon(years: pd.Series, horizon: int) -> pd.Series:
    completion_year = years.astype("Int64") + horizon
    completion_date = pd.to_datetime(completion_year.astype(str) + "-12-31", errors="coerce")
    return completion_date.le(pd.Timestamp(AS_OF_DATE))


def progression_flags(
    horizon: int, anchor_frame: pd.DataFrame, relation_frame: pd.DataFrame
) -> pd.DataFrame:
    if anchor_frame.empty:
        return pd.DataFrame()
    eligible = anchor_frame.loc[
        mature_for_horizon(anchor_frame["phase_i_year"], horizon)
    ].copy()
    followups = relation_frame.loc[
        relation_frame["phase_i_award_key"].isin(eligible["phase_i_award_key"])
        & relation_frame["award_year_gap"].le(horizon)
    ]
    observed = (
        followups.groupby(["phase_i_award_key", "phase_ii_program"]).size().unstack(fill_value=0)
        if not followups.empty
        else pd.DataFrame()
    )
    eligible = eligible.set_index("phase_i_award_key")
    for program in ("STTR", "SBIR"):
        keys = set(observed.index[observed[program].gt(0)]) if program in observed else set()
        eligible[f"observed_{program.lower()}_phase_ii"] = eligible.index.isin(keys)
    return eligible.reset_index()


horizon_rows = []
flags_by_horizon: dict[int, pd.DataFrame] = {}
sensitivity_flags_by_horizon: dict[int, pd.DataFrame] = {}
for horizon in FOLLOW_UP_HORIZONS:
    policy_frames = (
        (
            "primary — exclude multi-ID components",
            anchors,
            candidate_relations,
            flags_by_horizon,
        ),
        (
            "sensitivity — include multi-ID components",
            sensitivity_anchors,
            sensitivity_candidate_relations,
            sensitivity_flags_by_horizon,
        ),
    )
    for policy, anchor_frame, relation_frame, flag_store in policy_frames:
        flags = progression_flags(horizon, anchor_frame, relation_frame)
        flag_store[horizon] = flags
        if flags.empty:
            continue
        sttr_count = int(flags["observed_sttr_phase_ii"].sum())
        horizon_rows.append(
            {
                "identity-component policy": policy,
                "estimand": f"award-year gap <= {horizon} among {horizon}-horizon-mature firms",
                "horizon-mature first-Phase-I firms": len(flags),
                "right-censored firms excluded": len(anchor_frame) - len(flags),
                "STTR Phase II within gap bound": sttr_count,
                "descriptive STTR share among horizon-mature firms": sttr_count / len(flags),
                "SBIR Phase II within gap bound (separate)": int(
                    flags["observed_sbir_phase_ii"].sum()
                ),
            }
        )
horizon_summary = pd.DataFrame(horizon_rows)
horizon_summary


In [ ]:
primary_flags = flags_by_horizon.get(PRIMARY_HORIZON, pd.DataFrame()).copy()
sensitivity_primary_flags = sensitivity_flags_by_horizon.get(
    PRIMARY_HORIZON, pd.DataFrame()
).copy()
if primary_flags.empty:
    program_path_summary = pd.DataFrame()
    cross_program_diagnostic = pd.DataFrame()
else:
    sttr_seen = primary_flags["observed_sttr_phase_ii"]
    sbir_seen = primary_flags["observed_sbir_phase_ii"]
    primary_flags["observed_phase_ii_path"] = "none observed within horizon"
    primary_flags.loc[sttr_seen & ~sbir_seen, "observed_phase_ii_path"] = "STTR only"
    primary_flags.loc[~sttr_seen & sbir_seen, "observed_phase_ii_path"] = "SBIR only"
    primary_flags.loc[sttr_seen & sbir_seen, "observed_phase_ii_path"] = "both STTR and SBIR"
    program_path_summary = (
        primary_flags["observed_phase_ii_path"]
        .value_counts()
        .rename_axis("award-year gap <= 5 path among 5-horizon-mature firms")
        .rename("5-horizon-mature first-Phase-I firms")
        .to_frame()
    )
    cross_program_diagnostic = (
        primary_flags.groupby("cross_program_firm")["observed_sttr_phase_ii"]
        .agg(
            five_horizon_mature_firms="size",
            firms_with_sttr_phase_ii_gap_le_5="sum",
            descriptive_share_among_five_horizon_mature_firms="mean",
        )
        .rename_axis("firm has SBIR anywhere in snapshot")
    )
program_path_summary


In [ ]:
cross_program_diagnostic


## Timing and agency patterns among observed STTR follow-ons

Choose the first observed STTR Phase II with award-year gap ≤ 5 for each 5-horizon-mature firm. Report award-year lag and equality of normalized agency labels. Agency continuity has three states: `same`, `different`, and `unknown`; its share denominator includes only records where both normalized agencies are present. These are properties of observed award records, not transition mechanisms or agency effects. Cross-agency sequences can reflect unrelated projects held by the same firm.


In [ ]:
if primary_flags.empty:
    first_sttr_followups = pd.DataFrame()
    lag_distribution = pd.DataFrame()
    agency_progression = pd.DataFrame()
    agency_continuity = pd.DataFrame()
    agency_continuity_denominator = pd.DataFrame()
else:
    eligible_keys = set(primary_flags["phase_i_award_key"])
    first_sttr_followups = (
        candidate_relations.loc[
            candidate_relations["phase_i_award_key"].isin(eligible_keys)
            & candidate_relations["phase_ii_program"].eq("STTR")
            & candidate_relations["award_year_gap"].le(PRIMARY_HORIZON)
        ]
        .sort_values(
            ["phase_i_award_key", "phase_ii_year", "phase_ii_date", "phase_ii_award_key"],
            kind="stable",
            na_position="last",
        )
        .drop_duplicates("phase_i_award_key", keep="first")
    )
    lag_distribution = (
        first_sttr_followups["award_year_gap"]
        .value_counts()
        .sort_index()
        .rename_axis("award-year gap")
        .rename("firms")
        .to_frame()
    )
    known_agencies = first_sttr_followups["phase_i_agency_normalized"].notna() & (
        first_sttr_followups["phase_ii_agency_normalized"].notna()
    )
    same_agency = known_agencies & first_sttr_followups["phase_i_agency_normalized"].eq(
        first_sttr_followups["phase_ii_agency_normalized"]
    )
    different_agency = known_agencies & ~same_agency
    unknown_agency = ~known_agencies
    agency_continuity = pd.DataFrame(
        {
            "agency continuity": ["same", "different", "unknown"],
            "first STTR Phase II records": [
                int(same_agency.sum()),
                int(different_agency.sum()),
                int(unknown_agency.sum()),
            ],
        }
    ).set_index("agency continuity")
    known_agency_denominator = int(known_agencies.sum())
    agency_continuity_denominator = pd.DataFrame(
        [
            {
                "all first STTR Phase II records": len(first_sttr_followups),
                "both normalized agencies present (share denominator)": known_agency_denominator,
                "same-agency descriptive share among known": (
                    int(same_agency.sum()) / known_agency_denominator
                    if known_agency_denominator
                    else None
                ),
            }
        ]
    )
    agency_rows = []
    primary_flags["phase_i_agency_group"] = primary_flags["phase_i_agency_normalized"].fillna(
        "(unknown)"
    )
    for agency, group in primary_flags.groupby("phase_i_agency_group", dropna=False):
        count = int(group["observed_sttr_phase_ii"].sum())
        agency_rows.append(
            {
                "normalized Phase I agency": agency,
                "5-horizon-mature firms": len(group),
                "STTR Phase II firms with award-year gap <= 5": count,
                "descriptive share among 5-horizon-mature firms": count / len(group),
            }
        )
    agency_progression = pd.DataFrame(agency_rows).sort_values(
        ["5-horizon-mature firms", "normalized Phase I agency"], ascending=[False, True]
    )
lag_distribution


In [ ]:
display(agency_continuity, agency_continuity_denominator)


In [ ]:
agency_progression


## Phase III and effectiveness refusal

No STTR Phase II→III rate, latency distribution, survival probability, or effectiveness claim is produced. The award export has no Phase III rows in this pinned vintage, and the current transition marts drop the Phase II program and the Phase III SBIR/STTR code distinction. A pooled result would answer a different question.


In [ ]:
if source_verified:
    source_sttr_phase_iii_rows = int(
        (awards["program"].eq("STTR") & awards["phase_label"].eq("III")).sum()
    )
    phase_i_ii_disposition = "descriptive output available"
    phase_i_ii_reason = (
        "Program survives on both source rows; cross-program Phase II is separated; "
        "censoring is explicit."
    )
    source_phase_iii_reason = (
        "survival mart omits program and pinned award export has "
        f"{source_sttr_phase_iii_rows} STTR Phase III rows"
    )
else:
    source_sttr_phase_iii_rows = None
    phase_i_ii_disposition = "withheld — source unavailable or unverified"
    phase_i_ii_reason = (
        "Pinned award export is absent or failed byte/hash verification; no numerical "
        "Phase I→II description is available."
    )
    source_phase_iii_reason = (
        "survival mart omits program; pinned award export is absent or failed verification, "
        "so its STTR Phase III row count is unavailable"
    )
refusal_table = pd.DataFrame(
    [
        {
            "requested result": "STTR Phase I→II same-firm award-year-gap pattern",
            "disposition": phase_i_ii_disposition,
            "reason": phase_i_ii_reason,
        },
        {
            "requested result": "STTR Phase II→federal follow-on / Phase III rate",
            "disposition": "withheld",
            "reason": (
                "transition scores and pair marts omit program; no audited STTR-only "
                "denominator and outcome pair exists"
            ),
        },
        {
            "requested result": "STTR Phase II→III latency or survival",
            "disposition": "withheld",
            "reason": source_phase_iii_reason,
        },
        {
            "requested result": "STTR causal or program-effectiveness claim",
            "disposition": "refused",
            "reason": (
                "no comparison design, no project lineage, and incomplete Phase III "
                "observation"
            ),
        },
    ]
)
refusal_table


## Conclusion and decision

**Safe now:** describe program-preserved award-history sequences: the primary non-conflicted first-observed STTR Phase I firm components with an STTR Phase II at award-year gap ≤ N among N-horizon-mature firms, the corresponding multi-ID inclusion sensitivity, separately labelled cross-program paths, award-year lag, and three-state agency continuity. Keep every statement at the observed-record level.

**Not safe now:** report an STTR Phase II→III rate, latency, survival probability, transition-effectiveness comparison, or causal effect. The current transition outputs cannot distinguish STTR from SBIR, and the source award export cannot replace the missing Phase III outcome.

**Pipeline decision:** preserve nullable `phase_ii_program` and nullable `phase_iii_program`, together with the Phase III program-evidence source/method, through Phase II, Phase III, pair, score, and survival contracts before revisiting B3 for STTR. Populate `phase_iii_program` from `SR3`/`ST3` when that evidence exists; identification methods without authoritative SBIR/STTR evidence must retain `program = null`. The STTR-specific policy is to exclude unknown-program records from both numerator and denominator, report their count and coverage separately, and never impute program from the firm, paired Phase II, agency, or match method. An after-the-fact filter on the current marts is not sufficient.

This notebook does not reuse the three-agency commercialization-channel analysis and does not infer research-institution partner type.


In [ ]:
if primary_flags.empty:
    decision_summary = pd.DataFrame(
        [
            {
                "result": "No numerical result",
                "value": None,
                "limit": "Pinned awards input absent or invalid",
            }
        ]
    )
else:
    observed_count = int(primary_flags["observed_sttr_phase_ii"].sum())
    sensitivity_observed_count = int(
        sensitivity_primary_flags["observed_sttr_phase_ii"].sum()
    )
    decision_summary = pd.DataFrame(
        [
            {
                "result": "5-horizon-mature first-observed STTR Phase I firms",
                "value": len(primary_flags),
                "limit": "non-conflicted components; full five-calendar-year maturity",
            },
            {
                "result": "firms with STTR Phase II at award-year gap <= 5",
                "value": observed_count,
                "limit": "same firm, not project lineage",
            },
            {
                "result": "descriptive STTR share among 5-horizon-mature firms",
                "value": observed_count / len(primary_flags),
                "limit": "exploratory award-history description; not effectiveness",
            },
            {
                "result": "sensitivity 5-horizon-mature firms including multi-ID components",
                "value": len(sensitivity_primary_flags),
                "limit": "identity-inclusion sensitivity; not the primary denominator",
            },
            {
                "result": "sensitivity STTR share including multi-ID components",
                "value": sensitivity_observed_count / len(sensitivity_primary_flags),
                "limit": "selection-sensitive identity alternative; not effectiveness",
            },
            {
                "result": "STTR Phase II→III outcome",
                "value": None,
                "limit": "withheld because current marts drop program",
            },
        ]
    )
decision_summary


## Caveats and promotion checklist

- A firm can hold unrelated awards. Same-firm order is not scientific or contractual lineage.
- UEI and DUNS use canonical validation and sentinel removal before resolution. Exact alias components can still miss renamed firms or merge genuinely distinct entities that share an exact normalized name; components with multiple valid UEIs or DUNSes are reported and excluded from the primary description.
- Multi-ID components tend to be long-lived and award-rich, so primary exclusion selects on award history and may bias the descriptive share in an unknown direction. The inclusion sensitivity exposes the magnitude of this choice but cannot identify the correct firm history.
- Cross-program status uses the complete snapshot, including awards after the Phase I anchor. It is a diagnostic, not a baseline covariate or effect modifier.
- `Award Year` supports complete coverage but coarse timing. Proposal dates only resolve same-year order when coherent and strictly later; same-day relations are not later awards.
- Canonical phase normalization recognizes only exact I/II/III values after removing the `Phase` prefix. It excludes suffix-coded labels, so the 3,303 direct-to-Phase-II SBIR rows above are absent from the cross-program diagnostic rather than separately categorized.
- A missing later award is no observed record in this source and window, not failure or non-commercialization.
- Cross-horizon differences combine the follow-up bound and vintage composition because each horizon has a different maturity denominator.
- These are complete-source descriptive shares for the pinned export, so no sampling interval is reported; source coverage, identity error, and selection remain measurement limitations.

- [x] Question and decision are explicit.
- [x] Data snapshot, SHA-256, population, grain, keys, exclusions, assumptions, and seed are recorded.
- [x] Cross-program firms and right censoring are explicit.
- [x] Phase III and causal/effectiveness claims are refused.
- [x] Destination tier is `exploratory` and non-citable.
- [ ] Nullable program provenance and an explicit unknown-program policy are preserved through the Phase II/III transition contracts.
- [ ] Firm-component logic has a public versioned primitive suitable for promotion.
- [ ] A promoted study freezes inputs, lineage rules, estimand, validation, and claim scope.
- [x] Outputs and execution counts are cleared before commit.

Until those unchecked items are completed through explicit promotion work, the numerical tables remain exploratory and non-citable.
